# Notebook 03 — Entrenamiento y Comparación de Modelos

**Proyecto:** IntApp — Predicción del riesgo de lesión deportiva  
**Autor:** Roberto Franco  
**Director:** Javier Sánchez  
**Universidad:** Universidad Europea de Madrid  

---


## 1. Introducción

En este notebook entrenamos y comparamos **tres algoritmos de Machine Learning** para predecir el nivel de riesgo de lesión de miembro inferior en deportistas:

- **Random Forest**: conjunto de árboles de decisión, robusto ante outliers.
- **Regresión Logística**: modelo lineal, interpretable y rápido.
- **Gradient Boosting**: ensamblado secuencial, habitualmente el más preciso.

La métrica principal es el **F1-score macro** sobre las tres clases (bajo / medio / alto). Se aplica además una **estrategia clínica asimétrica** para minimizar los falsos negativos en la clase 'alto': en prevención de lesiones, no detectar un deportista de alto riesgo es mucho más grave que sobre-alertar a uno sano.

> **Nota:** la categoría `no_concluyente` (NRS > 5) se gestiona como regla pre-modelo: si el deportista tiene dolor agudo activo, la app muestra el aviso directamente sin pasar por el clasificador.


## 2. Configuración e importaciones


In [1]:
import sys
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

# Añadir la raíz del proyecto al path
RAIZ_PROYECTO = Path(os.getcwd()).parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.generador_datos import generar_dataset
from src.preprocesador import preprocesar
from src.modelo import (
    dividir_datos,
    evaluar_modelo,
    comparar_modelos,
    guardar_modelo,
    entrenar_gradient_boosting,
    calibrar_umbral_alto,
    validar_cruzada,
    CalibradorUmbralAlto,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

print(f'Raíz del proyecto: {RAIZ_PROYECTO}')
print('Importaciones correctas.')


Raíz del proyecto: /Users/__robeerr/Programacion_Local/IntApp v2
Importaciones correctas.


## 3. Generación y preprocesamiento de datos

Generamos el dataset combinando **5 semillas × 1.000 muestras = 5.000 deportistas**. Usar varias semillas aumenta la variabilidad y mejora la generalización del modelo.

In [2]:
print('Generando dataset combinado (5 semillas x 1000 muestras)...')
dfs = []
for seed in [42, 123, 2025, 819, 356]:
    df_seed = generar_dataset(n_deportistas=1000, semilla=seed)
    dfs.append(df_seed)
df_crudo = pd.concat(dfs, ignore_index=True)

# 'no_concluyente' es una regla clínica pre-modelo (NRS > 5).
# Se filtra aquí: la app la gestiona antes de llamar al clasificador.
df_crudo = df_crudo[df_crudo['riesgo_lesion'] != 'no_concluyente'].copy()
df_crudo = df_crudo.reset_index(drop=True)

print(f'Dataset: {df_crudo.shape[0]} filas x {df_crudo.shape[1]} columnas')
print('\nDistribución de clases:')
print(df_crudo['riesgo_lesion'].value_counts())
df_crudo.head(3)

Generando dataset combinado (5 semillas x 1000 muestras)...
Generando dataset sintético v2.3 con 1000 deportistas (semilla=42)...
  [1/5] Bloque contexto...
  [2/5] Bloque fuerza...
  [3/5] Bloque movilidad...
  [4/5] Bloque control...
  [5/5] Inyectando casos frontera...
  Aplicando reglas v2.3 y calculando score de confianza...

Distribución de riesgo:
  bajo            :  30.6 %
  medio           :  23.5 %
  alto            :  42.1 %
  no_concluyente  :   3.8 %

Distribución de confianza:
  alta            :  20.6 %
  media           :  41.4 %
  baja            :  38.0 %

Dataset generado: 1000 filas × 36 columnas.

Generando dataset sintético v2.3 con 1000 deportistas (semilla=123)...
  [1/5] Bloque contexto...
  [2/5] Bloque fuerza...
  [3/5] Bloque movilidad...
  [4/5] Bloque control...
  [5/5] Inyectando casos frontera...
  Aplicando reglas v2.3 y calculando score de confianza...

Distribución de riesgo:
  bajo            :  28.2 %
  medio           :  25.4 %
  alto            :

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,peso_corporal,nivel_actividad,historial_lesional,dolor_percibido_nrs,hooper_index,riesgo_lesion,score_total,confianza_score,confianza_categoria,reglas_activadas
0,334.6,309.0,96.1,153.6,116.8,77.0,89.5,77.8,129.9,126.1,...,64.7,activo,2,5,26,medio,17.5,68.0,media,M3
1,440.5,513.7,276.5,272.4,157.9,161.1,134.0,123.1,244.2,241.2,...,77.1,activo,2,1,19,bajo,8.0,88.0,alta,—
2,80.0,80.0,90.2,103.5,40.0,40.0,25.0,25.0,72.0,78.1,...,69.4,sedentario,4,0,15,alto,46.0,28.0,baja,"A1_der,A2_der,M7"


In [3]:
# Pipeline de preprocesamiento v2.3 (6 pasos):
# 1. calcular_ratios           → N → N/kg; ratios H:Q y ADD/ABD
# 2. calcular_ratios_normativos → N/kg → ratio_ref por perfil (edad×género×actividad)
# 3. calcular_asimetrias       → asimetría bilateral (%) por par _der/_izq
# 4. codificar_categoricas     → genero binario, nivel_actividad ordinal 0-3
# 5. eliminar_columnas_aux     → elimina score_total, confianza_*, reglas_activadas
# 6. normalizar                → StandardScaler sobre 47 columnas continuas
# → Salida: 57 columnas (56 features + riesgo_lesion)
df_procesado, scaler = preprocesar(df_crudo)

print(f'Dataset preprocesado: {df_procesado.shape[0]} filas x {df_procesado.shape[1]} columnas')
print(f'Features para el modelo: {df_procesado.shape[1] - 1}')
print(f'NaN en el dataset: {df_procesado.isnull().sum().sum()}')

# Guardar datos procesados
DIR_PROCESADOS = RAIZ_PROYECTO / 'datos' / 'procesados'
DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)
df_procesado.to_csv(DIR_PROCESADOS / 'dataset_procesado.csv', index=False)
print(f'Dataset procesado guardado.')


2026-06-06 11:48:58 | INFO | src.preprocesador | Inicio preprocesamiento v2.3. Filas: 4826 | Columnas: 36
2026-06-06 11:48:58 | INFO | src.preprocesador | Paso 1/6: ratios clínicos y N/kg calculados.
2026-06-06 11:48:58 | INFO | src.preprocesador | Paso 2/6: ratios normativos por perfil calculados.
2026-06-06 11:48:58 | INFO | src.preprocesador | Paso 3/6: asimetrías bilaterales calculadas.
2026-06-06 11:48:58 | INFO | src.preprocesador | Paso 4/6: variables categóricas codificadas.
2026-06-06 11:48:58 | INFO | src.preprocesador | Eliminando columnas auxiliares: ['score_total', 'confianza_score', 'confianza_categoria', 'reglas_activadas']
2026-06-06 11:48:58 | INFO | src.preprocesador | Paso 5/6: columnas auxiliares eliminadas.
2026-06-06 11:48:58 | INFO | src.preprocesador | StandardScaler ajustado sobre 47 columnas.
2026-06-06 11:48:58 | INFO | src.preprocesador | Paso 6/6: normalización aplicada.
2026-06-06 11:48:58 | INFO | src.preprocesador | Preprocesamiento finalizado. Filas: 48

Dataset preprocesado: 4826 filas x 57 columnas
Features para el modelo: 56
NaN en el dataset: 0
Dataset procesado guardado.


## 4. División en entrenamiento y test

- **Entrenamiento (80%):** datos que el modelo usa para aprender.
- **Test (20%):** datos que el modelo **nunca ha visto**, usados para las métricas finales.

Se usa `stratify` para preservar la proporción de clases en ambos conjuntos.


In [4]:
X_train, X_test, y_train, y_test = dividir_datos(
    df_procesado,
    columna_objetivo='riesgo_lesion',
    test_size=0.20,
    semilla=42,
)

print(f'Train: {len(X_train)} muestras | Test: {len(X_test)} muestras | Features: {X_train.shape[1]}')
print()
dist_train = y_train.value_counts(normalize=True).mul(100).round(1)
dist_test  = y_test.value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({'Train (%)': dist_train, 'Test (%)': dist_test}).reindex(['bajo','medio','alto'])


2026-06-06 11:48:58 | INFO | src.modelo | División completada — Entrenamiento: 3860 muestras | Prueba: 966 muestras
2026-06-06 11:48:58 | INFO | src.modelo | Distribución en entrenamiento:
riesgo_lesion
alto     0.433
bajo     0.303
medio    0.264


Train: 3860 muestras | Test: 966 muestras | Features: 56



,Train (%),Test (%)
riesgo_lesion,,
bajo,30.3,30.3
medio,26.4,26.4
alto,43.3,43.3


## 4b. Distribución del dataset de entrenamiento

Figura para la memoria del TFM: distribución real de clases sobre el dataset completo preprocesado (n=4.826).

In [ ]:
# ── Figura: Distribución del dataset de entrenamiento (n=4826) ──────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIGURAS_DIR_NB03 = RAIZ_PROYECTO / 'figuras'
FIGURAS_DIR_NB03.mkdir(parents=True, exist_ok=True)

ORDEN_RIESGO_3  = ['bajo', 'medio', 'alto']
COLORES_RIESGO_3 = {'bajo': '#2ecc71', 'medio': '#f39c12', 'alto': '#e74c3c'}

conteo = df_procesado['riesgo_lesion'].value_counts().reindex(ORDEN_RIESGO_3)
pcts   = conteo / conteo.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax_bar = axes[0]
barras = ax_bar.bar(
    ORDEN_RIESGO_3,
    conteo.values,
    color=[COLORES_RIESGO_3[n] for n in ORDEN_RIESGO_3],
    edgecolor='white', linewidth=1.5, width=0.5
)
for barra, n, pct in zip(barras, conteo.values, pcts.values):
    ax_bar.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + max(conteo.values) * 0.01,
        f'{n}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
ax_bar.set_title('Distribución de niveles de riesgo', fontsize=12, fontweight='bold')
ax_bar.set_xlabel('Nivel de riesgo de lesión', fontsize=10)
ax_bar.set_ylabel('Número de deportistas', fontsize=10)
ax_bar.set_xticklabels(['Bajo', 'Medio', 'Alto'], fontsize=11)
ax_bar.set_ylim(0, max(conteo.values) * 1.15)

ax_pie = axes[1]
wedges, texts, autotexts = ax_pie.pie(
    conteo.values,
    labels=['Bajo', 'Medio', 'Alto'],
    autopct='%1.1f%%',
    colors=[COLORES_RIESGO_3[n] for n in ORDEN_RIESGO_3],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    pctdistance=0.78,
)
for t in texts: t.set_fontsize(11)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold'); at.set_color('white')
ax_pie.set_title('Proporción de cada nivel de riesgo', fontsize=12, fontweight='bold')

fig.suptitle(
    f'Variable objetivo: riesgo_lesion — Dataset de entrenamiento (N = {len(df_procesado)} deportistas)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()

ruta_fig = FIGURAS_DIR_NB03 / 'distribucion_riesgo_entrenamiento.png'
fig.savefig(ruta_fig, dpi=150, bbox_inches='tight')
print(f'✓ Figura guardada: {ruta_fig}')
plt.show()


## 5. Entrenamiento de los modelos

Se usan hiperparámetros optimizados:
- **Random Forest:** 500 árboles, profundidad 12, pesos balanceados.
- **Regresión Logística:** regularización C=0.5, pesos balanceados.
- **Gradient Boosting:** 400 estimadores, learning rate 0.05, profundidad 4, subsample 0.8.


In [5]:
print('=' * 55)
print('Entrenando 1/3: Random Forest')
print('=' * 55)
t0 = time.time()
modelo_rf = RandomForestClassifier(
    n_estimators=500, max_depth=12, min_samples_leaf=2,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1,
)
modelo_rf.fit(X_train, y_train)
tiempo_rf = time.time() - t0
print(f'Tiempo: {tiempo_rf:.2f}s')


Entrenando 1/3: Random Forest
Tiempo: 0.52s


In [6]:
print('=' * 55)
print('Entrenando 2/3: Regresion Logistica')
print('=' * 55)
t0 = time.time()
modelo_rl = LogisticRegression(
    C=0.5, class_weight='balanced',
    max_iter=2000, solver='lbfgs', random_state=42,
)
modelo_rl.fit(X_train, y_train)
tiempo_rl = time.time() - t0
print(f'Tiempo: {tiempo_rl:.2f}s')


Entrenando 2/3: Regresion Logistica
Tiempo: 0.10s


In [7]:
print('=' * 55)
print('Entrenando 3/3: Gradient Boosting')
print('=' * 55)
t0 = time.time()
pesos_balanced = compute_sample_weight('balanced', y_train)
modelo_gb = GradientBoostingClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=4,
    subsample=0.8, min_samples_leaf=2, random_state=42,
)
modelo_gb.fit(X_train, y_train, sample_weight=pesos_balanced)
tiempo_gb = time.time() - t0
print(f'Tiempo: {tiempo_gb:.2f}s')
print()
pd.DataFrame({
    'Modelo': ['Random Forest', 'Regresion Logistica', 'Gradient Boosting'],
    'Tiempo (s)': [round(tiempo_rf, 2), round(tiempo_rl, 2), round(tiempo_gb, 2)],
})


Entrenando 3/3: Gradient Boosting
Tiempo: 29.69s



,Modelo,Tiempo (s)
0,Random Forest,0.52
1,Regresion Logistica,0.10
2,Gradient Boosting,29.69


## 6. Evaluacion de los modelos

Evaluamos cada modelo sobre el conjunto de test. La metrica principal es el **F1-score macro** (promedio no ponderado de las 3 clases).


### 6.1 Random Forest


In [8]:
resultado_rf = evaluar_modelo(modelo_rf, X_test, y_test, nombre='Random Forest')


2026-06-06 11:49:29 | INFO | src.modelo | Evaluando modelo: Random Forest
2026-06-06 11:49:29 | INFO | src.modelo | Random Forest — Accuracy: 0.8333 | F1 macro: 0.8189
2026-06-06 11:49:29 | INFO | src.modelo | Matriz de confusión guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/confusion_random_forest.png



  RESULTADOS — RANDOM FOREST

Muestras de prueba: 966

Informe de clasificación:
              precision    recall  f1-score   support

        bajo       0.85      0.84      0.84       293
       medio       0.72      0.71      0.72       255
        alto       0.88      0.91      0.89       418

    accuracy                           0.83       966
   macro avg       0.82      0.82      0.82       966
weighted avg       0.83      0.83      0.83       966

Exactitud (accuracy)     : 0.8333
Precisión macro          : 0.8204
Recall macro             : 0.8176
F1-score macro           : 0.8189
Recall clase 'alto'      : 0.9067  ← prioridad clínica



### 6.2 Regresion Logistica


In [9]:
resultado_rl = evaluar_modelo(modelo_rl, X_test, y_test, nombre='Regresion Logistica')


2026-06-06 11:49:29 | INFO | src.modelo | Evaluando modelo: Regresion Logistica
2026-06-06 11:49:29 | INFO | src.modelo | Regresion Logistica — Accuracy: 0.7505 | F1 macro: 0.7285



  RESULTADOS — REGRESION LOGISTICA

Muestras de prueba: 966

Informe de clasificación:
              precision    recall  f1-score   support

        bajo       0.73      0.71      0.72       293
       medio       0.56      0.61      0.59       255
        alto       0.89      0.86      0.88       418

    accuracy                           0.75       966
   macro avg       0.73      0.73      0.73       966
weighted avg       0.76      0.75      0.75       966

Exactitud (accuracy)     : 0.7505
Precisión macro          : 0.7291
Recall macro             : 0.7288
F1-score macro           : 0.7285
Recall clase 'alto'      : 0.8612  ← prioridad clínica



2026-06-06 11:49:29 | INFO | src.modelo | Matriz de confusión guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/confusion_regresion_logistica.png


### 6.3 Gradient Boosting


In [10]:
resultado_gb = evaluar_modelo(modelo_gb, X_test, y_test, nombre='Gradient Boosting')


2026-06-06 11:49:29 | INFO | src.modelo | Evaluando modelo: Gradient Boosting
2026-06-06 11:49:29 | INFO | src.modelo | Gradient Boosting — Accuracy: 0.8747 | F1 macro: 0.8657



  RESULTADOS — GRADIENT BOOSTING

Muestras de prueba: 966

Informe de clasificación:
              precision    recall  f1-score   support

        bajo       0.88      0.92      0.90       293
       medio       0.77      0.80      0.78       255
        alto       0.94      0.89      0.92       418

    accuracy                           0.87       966
   macro avg       0.86      0.87      0.87       966
weighted avg       0.88      0.87      0.88       966

Exactitud (accuracy)     : 0.8747
Precisión macro          : 0.8632
Recall macro             : 0.8692
F1-score macro           : 0.8657
Recall clase 'alto'      : 0.8900  ← prioridad clínica



2026-06-06 11:49:29 | INFO | src.modelo | Matriz de confusión guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/confusion_gradient_boosting.png


## 7. Comparacion entre modelos

Tabla resumen y grafico de barras con el F1-score macro de cada modelo.


In [11]:
CLASES = ['bajo', 'medio', 'alto']

resultados_todos = {
    'Random Forest':       resultado_rf,
    'Regresion Logistica': resultado_rl,
    'Gradient Boosting':   resultado_gb,
}

# comparar_modelos devuelve un DataFrame con las metricas y genera el grafico
df_comparacion = comparar_modelos(resultados_todos)

# Identificar el mejor modelo
mejor_nombre = df_comparacion['F1 macro'].idxmax()
mapa_modelos = {
    'Random Forest':       modelo_rf,
    'Regresion Logistica': modelo_rl,
    'Gradient Boosting':   modelo_gb,
}
mejor_modelo_estandar = mapa_modelos[mejor_nombre]
print(f'Mejor modelo por F1-score macro: {mejor_nombre}')



  COMPARACIÓN DE MODELOS
                     Accuracy  Precisión macro  Recall macro  F1 macro
Modelo                                                                
Random Forest          0.8333           0.8204        0.8176    0.8189
Regresion Logistica    0.7505           0.7291        0.7288    0.7285
Gradient Boosting      0.8747           0.8632        0.8692    0.8657

Mejor modelo por F1 macro: Gradient Boosting (0.8657)



2026-06-06 11:49:29 | INFO | src.modelo | Figura de comparación guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/comparacion_modelos.png


Mejor modelo por F1-score macro: Gradient Boosting


## 7b. Validación cruzada (5-fold estratificada)

Antes de la estrategia clínica asimétrica, ejecutamos una **validación cruzada de 5 folds** sobre el conjunto de entrenamiento.  
Esto estima la varianza del rendimiento más allá del split 80/20 único y detecta posible sobreajuste.


In [12]:
df_cv = validar_cruzada(X_train, y_train, semilla=42, n_folds=5)
df_cv



--- Validación cruzada (5-fold estratificada) ---
  Fold 1/5 completado.
  Fold 2/5 completado.
  Fold 3/5 completado.
  Fold 4/5 completado.
  Fold 5/5 completado.

  VALIDACIÓN CRUZADA (5-FOLD) — RESULTADOS
                     F1_media  F1_std  Recall_alto_media  Recall_alto_std
Modelo                                                                   
Random Forest          0.8151  0.0250             0.9120           0.0226
Regresión Logística    0.7239  0.0276             0.8348           0.0161
Gradient Boosting      0.8759  0.0154             0.9222           0.0103



,F1_media,F1_std,Recall_alto_media,Recall_alto_std
Modelo,,,,
Random Forest,0.8151,0.0250,0.9120,0.0226
Regresión Logística,0.7239,0.0276,0.8348,0.0161
Gradient Boosting,0.8759,0.0154,0.9222,0.0103


## 8. Estrategia clínica asimétrica

El modelo estándar clasifica incorrectamente como 'bajo' o 'medio' a algunos deportistas de alto riesgo.  
En prevención de lesiones, **este es el error más peligroso** (falso negativo).

Se aplica una estrategia en dos pasos:

1. **Pesos de clase en el entrenamiento** — `entrenar_gradient_boosting(..., peso_extra_alto=2.0, peso_extra_medio=1.8)` penaliza los errores en 'alto' y 'medio' de forma automática y reproducible.
2. **Umbral de decisión calibrado** — `calibrar_umbral_alto()` encuentra el umbral que maximiza el recall de 'alto' sujeto a **dos restricciones simultáneas**: F1 macro ≥ 0.58 **y** precisión 'alto' ≥ 0.60.

> *Justificación: el coste de un falso negativo (no detectar un deportista en riesgo) puede ser una lesión grave.  
> La restricción de precisión evita umbrales agresivos (p.ej. 0.10) que generarían una tasa de falsas alarmas clínicamente inaceptable.*

In [13]:
# Val split interno para calibrar el umbral (nunca toca X_test)
X_tr_gb, X_val_gb, y_tr_gb, y_val_gb = train_test_split(
    X_train, y_train, test_size=0.20, random_state=42, stratify=y_train
)
print(f'Split calibración — Entrenamiento: {len(X_tr_gb)} | Validación: {len(X_val_gb)}')

# Entrenar GB clínico con peso extra para 'alto' y 'medio'
print('\nEntrenando GB clínico (peso_extra_alto=2.0, peso_extra_medio=1.8)...')
t0 = time.time()
modelo_gb_cal_base = entrenar_gradient_boosting(X_tr_gb, y_tr_gb, semilla=42, peso_extra_alto=2.0, peso_extra_medio=1.8)
print(f'Entrenado en {time.time()-t0:.2f}s')

# Calibrar umbral sobre val set con restricciones clínicas
umbral_optimo = calibrar_umbral_alto(
    modelo_gb_cal_base, X_val_gb, y_val_gb,
    min_f1_macro=0.58, min_precision_alto=0.60,
)
print(f'Umbral óptimo seleccionado: {umbral_optimo:.2f}')

2026-06-06 11:52:00 | INFO | src.modelo | Entrenando Gradient Boosting (n_estimators=400, lr=0.05, subsample=0.8, peso_extra_alto=2.0, peso_extra_medio=1.8)...
2026-06-06 11:52:00 | INFO | src.modelo | Pesos de muestra. Clases: ['alto' 'bajo' 'medio'] | Pesos únicos: [1.1009 1.5398 2.2706]


Split calibración — Entrenamiento: 3088 | Validación: 772

Entrenando GB clínico (peso_extra_alto=2.0, peso_extra_medio=1.8)...


2026-06-06 11:52:31 | INFO | src.modelo | Gradient Boosting entrenado correctamente.


Entrenado en 30.24s

  Calibración de umbral para 'alto' (val set):
    Umbral | Recall alto | Prec alto |   F1 macro |   OK
  ----------------------------------------------------
      0.50 |      0.9162 |    0.9503 |     0.8808 |    ✓
      0.48 |      0.9162 |    0.9415 |     0.8766 |    ✓
      0.46 |      0.9192 |    0.9417 |     0.8778 |    ✓
      0.44 |      0.9222 |    0.9419 |     0.8789 |    ✓
      0.42 |      0.9281 |    0.9394 |     0.8814 |    ✓
      0.40 |      0.9311 |    0.9367 |     0.8812 |    ✓
      0.38 |      0.9311 |    0.9367 |     0.8812 |    ✓
      0.36 |      0.9311 |    0.9339 |     0.8798 |    ✓
      0.34 |      0.9341 |    0.9286 |     0.8781 |    ✓
      0.32 |      0.9341 |    0.9231 |     0.8753 |    ✓
      0.30 |      0.9431 |    0.9211 |     0.8774 |    ✓
      0.28 |      0.9461 |    0.9186 |     0.8771 |    ✓
      0.26 |      0.9581 |    0.9143 |     0.8789 |    ✓
      0.24 |      0.9611 |    0.9093 |     0.8789 |    ✓
      0.22 |      0.96

In [14]:
# Wrapper sklearn con umbral calibrado
modelo_gb_cal = CalibradorUmbralAlto(modelo_gb_cal_base, umbral_alto=umbral_optimo)

# Comparativa: GB estándar vs GB calibrado
CLASES = ['bajo', 'medio', 'alto']
print('=' * 62)
print('  GB ESTÁNDAR  vs  GB CALIBRADO (umbral calibrado con restricciones)')
print('=' * 62)
print('\n--- GB estándar ---')
print(classification_report(
    y_test, modelo_gb.predict(X_test),
    labels=CLASES, target_names=CLASES, zero_division=0
))
print(f'--- GB Calibrado (umbral alto = {umbral_optimo:.2f}) ---')
print(classification_report(
    y_test, modelo_gb_cal.predict(X_test),
    labels=CLASES, target_names=CLASES, zero_division=0
))

# Métrica clave: falsos negativos alto → bajo
y_pred_gb_std = modelo_gb.predict(X_test)
y_pred_cal    = modelo_gb_cal.predict(X_test)
fn_estandar  = sum(1 for r, p in zip(y_test, y_pred_gb_std) if r == 'alto' and p == 'bajo')
fn_calibrado = sum(1 for r, p in zip(y_test, y_pred_cal)    if r == 'alto' and p == 'bajo')
total_alto   = sum(y_test == 'alto')

print('=' * 62)
print('  MÉTRICA CLÍNICA CLAVE: falsos negativos alto → bajo')
print('=' * 62)
print(f'  GB estándar  : {fn_estandar} de {total_alto} casos ({fn_estandar/total_alto*100:.1f}%)')
print(f'  GB Calibrado : {fn_calibrado} de {total_alto} casos ({fn_calibrado/total_alto*100:.1f}%)')
print(f'\n  Reducción de falsos negativos: {fn_estandar - fn_calibrado} casos menos sin detectar')


  GB ESTÁNDAR  vs  GB CALIBRADO (umbral calibrado con restricciones)

--- GB estándar ---
              precision    recall  f1-score   support

        bajo       0.88      0.92      0.90       293
       medio       0.77      0.80      0.78       255
        alto       0.94      0.89      0.92       418

    accuracy                           0.87       966
   macro avg       0.86      0.87      0.87       966
weighted avg       0.88      0.87      0.88       966

--- GB Calibrado (umbral alto = 0.12) ---
              precision    recall  f1-score   support

        bajo       0.91      0.87      0.89       293
       medio       0.82      0.67      0.74       255
        alto       0.85      0.96      0.90       418

    accuracy                           0.86       966
   macro avg       0.86      0.84      0.84       966
weighted avg       0.86      0.86      0.86       966

  MÉTRICA CLÍNICA CLAVE: falsos negativos alto → bajo
  GB estándar  : 6 de 418 casos (1.4%)
  GB Calibrad

### 8.1 Intervalo de confianza bootstrap (recall_alto)

El test set tiene ≈ 400 muestras con ~80 de clase 'alto'. Un solo recall puntual no informa de su incertidumbre. El bootstrap remuestrea 1 000 veces con reemplazamiento y devuelve el IC 95% por percentiles, sin asumir normalidad.

In [15]:
from src.modelo import bootstrap_ic

_recall_fn = lambda yt, yp: recall_score(yt, yp, labels=['alto'], average='macro', zero_division=0)
ic_lo, ic_hi = bootstrap_ic(
    y_test, modelo_gb_cal.predict(X_test), _recall_fn,
    n_iter=1000, nivel_confianza=0.95, semilla=42,
)
print(f'Recall alto (test)  : {_recall_fn(y_test, modelo_gb_cal.predict(X_test)):.4f}')
print(f'IC 95% bootstrap    : [{ic_lo:.4f}, {ic_hi:.4f}]')
print(f'Amplitud del IC     : {ic_hi - ic_lo:.4f}  '
      f'(amplitud alta → test set pequeño, interpretar con cautela)')

Recall alto (test)  : 0.9641
IC 95% bootstrap    : [0.9443, 0.9811]
Amplitud del IC     : 0.0368  (amplitud alta → test set pequeño, interpretar con cautela)


## 9. Guardado del modelo final

Se guarda **`CalibradorUmbralAlto`** como modelo definitivo: un wrapper que encapsula el GB clínico junto con el umbral calibrado. Es el que se carga en la app Streamlit para evaluar nuevos deportistas — `predict()` aplica automáticamente el umbral óptimo.

In [16]:
DIR_MODELOS = RAIZ_PROYECTO / 'modelos'
DIR_MODELOS.mkdir(parents=True, exist_ok=True)

# Modelo principal: CalibradorUmbralAlto (wrapper GB clínico + umbral calibrado)
guardar_modelo(modelo_gb_cal, DIR_MODELOS / 'mejor_modelo.pkl')
print('Guardado: mejor_modelo.pkl  →  CalibradorUmbralAlto (GB clínico, umbral calibrado)')

# Modelos estándar para comparación y análisis SHAP
joblib.dump(modelo_rf, DIR_MODELOS / 'modelo_rf.joblib')
joblib.dump(modelo_rl, DIR_MODELOS / 'modelo_rl.joblib')
joblib.dump(modelo_gb, DIR_MODELOS / 'modelo_gb.joblib')
print('Guardados: modelo_rf.joblib, modelo_rl.joblib, modelo_gb.joblib')

# Scaler necesario para preprocesar nuevos deportistas en la app
joblib.dump(scaler, DIR_MODELOS / 'scaler.pkl')
print('Guardado: scaler.pkl')

# Resumen final
y_pred_cal_test = modelo_gb_cal.predict(X_test)
f1_cal    = f1_score(y_test, y_pred_cal_test, average='macro', labels=CLASES, zero_division=0)
rec_alto  = recall_score(y_test, y_pred_cal_test, labels=['alto'], average='macro', zero_division=0)
prec_alto = precision_score(y_test, y_pred_cal_test, labels=['alto'], average='macro', zero_division=0)
fn_cal    = sum(1 for r, p in zip(y_test, y_pred_cal_test) if r == 'alto' and p == 'bajo')

print()
print('=== RESUMEN FINAL ===')
print(f'Modelo seleccionado    : CalibradorUmbralAlto (GB clínico)')
print(f'Umbral clase alto      : {umbral_optimo:.2f}  (precision≥0.60, F1≥0.58)')
print(f'F1-macro test          : {f1_cal:.4f}')
print(f'Recall clase alto      : {rec_alto:.4f}')
print(f'Precisión clase alto   : {prec_alto:.4f}')
print(f'Falsos neg. alto→bajo  : {fn_cal} de {total_alto} ({fn_cal/total_alto*100:.1f}%)')
print(f'CV 5-fold GB (F1)      : {df_cv.loc["Gradient Boosting","F1_media"]:.4f} ± {df_cv.loc["Gradient Boosting","F1_std"]:.4f}')
print(f'CV 5-fold GB (Rec.alto): {df_cv.loc["Gradient Boosting","Recall_alto_media"]:.4f} ± {df_cv.loc["Gradient Boosting","Recall_alto_std"]:.4f}')
print(f'Dataset                : 5 semillas x 1.000 = 5.000 muestras')

2026-06-06 11:52:32 | INFO | src.modelo | Modelo guardado en: /Users/__robeerr/Programacion_Local/IntApp v2/modelos/mejor_modelo.pkl


Guardado: mejor_modelo.pkl  →  CalibradorUmbralAlto (GB clínico, umbral calibrado)
Guardados: modelo_rf.joblib, modelo_rl.joblib, modelo_gb.joblib
Guardado: scaler.pkl

=== RESUMEN FINAL ===
Modelo seleccionado    : CalibradorUmbralAlto (GB clínico)
Umbral clase alto      : 0.12  (precision≥0.60, F1≥0.58)
F1-macro test          : 0.8444
Recall clase alto      : 0.9641
Precisión clase alto   : 0.8520
Falsos neg. alto→bajo  : 4 de 418 (1.0%)
CV 5-fold GB (F1)      : 0.8759 ± 0.0154
CV 5-fold GB (Rec.alto): 0.9222 ± 0.0103
Dataset                : 5 semillas x 1.000 = 5.000 muestras
